# Shortcut or shelter?

## Finding the policy boundary in a noisy maze

A short corridor reaches the goal quickly, but lateral execution errors can
enter hazards above or below it. A longer southern route faces the same
actuator noise; a wall and the grid boundary convert its lateral errors into
delays rather than danger.

We ask three deliberately separate questions:

1. At what intended-action reliability does the **exact** optimal policy
   switch routes?
2. How do Q-learning, SARSA, and Expected SARSA approach that boundary?
3. Does continuing exploration after training change which route should be
   valued?

Two consequence laws use exactly the same topology. A recoverable hazard
charges a penalty and lets the episode continue; a lethal hazard charges a
large penalty and ends the episode. The lethal case is not softened to make
a prettier plot: its exact transition really is compressed close to perfect
control.

This notebook is a controlled study of backup targets, not an algorithm
leaderboard. `QUICK=True` is a mechanics check. Publication claims require
the versioned full configurations, their complete seed panels, and the saved
Protocol-v2 manifests.


In [ ]:
from __future__ import annotations

from dataclasses import replace
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rllab.environments import RiskyCorridorEnv
from rllab.experiments import (
    Experiment,
    ExperimentConfig,
    ExperimentResult,
    ExecutionSpec,
    RunStore,
    estimate_run,
)
from rllab.metrics import bootstrap_confidence_interval
from rllab.theory import (
    epsilon_soft_value_iteration,
    policy_evaluation as exact_policy_evaluation,
    value_iteration,
)
from rllab.visualization import plot_maze, plot_policy, plot_transition_noise

SMOKE = os.environ.get("RL_LAB_NOTEBOOK_SMOKE") == "1"
QUICK = True
SHOW_PROGRESS = False  # reliable in every Jupyter frontend; the CLI has live progress
GAMMA = 0.98
PERSISTENT_EPSILON = 0.10
INITIAL_Q = 8.0
RECOVERABLE_PENALTY = -0.50  # about eight ordinary movement costs
LETHAL_PENALTY = -8.0
METHOD_ORDER = ("q_learning", "sarsa", "expected_sarsa")
METHOD_LABELS = {
    "q_learning": "Q-learning",
    "sarsa": "SARSA",
    "expected_sarsa": "Expected SARSA",
}
METHOD_COLORS = {
    "q_learning": "#0072B2",
    "sarsa": "#D55E00",
    "expected_sarsa": "#009E73",
}
DISAGREEMENT_POINTS = {"recoverable": 0.825, "lethal": 1.0}
EXISTING_RUNS: dict[str, str | Path | None] = {
    "recoverable": None,
    "lethal": None,
    "annealed": None,
}

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "rllab").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the rl-lab repository.")

REPO_ROOT = find_repo_root(Path.cwd())
configured_results = os.environ.get("RL_LAB_NOTEBOOK_RESULTS")
RESULTS_DIR = Path(configured_results) if configured_results else REPO_ROOT / "results"
N_RESAMPLES = 50 if SMOKE else (500 if QUICK else 2_000)
REPRESENTATIVE_SEED = 0
RUN_PROFILE = "SMOKE" if SMOKE else ("QUICK" if QUICK else "FULL")
EMPIRICAL_TITLE = "" if RUN_PROFILE == "FULL" else f" — {RUN_PROFILE}: NOT FOR INFERENCE"
if RUN_PROFILE != "FULL":
    print(
        f"*** {RUN_PROFILE} PROFILE: validates mechanics only; "
        "do not publish or interpret learned boundaries. ***"
    )

available_styles = set(plt.style.available)
plot_style = next(
    (
        style
        for style in ("seaborn-v0_8-whitegrid", "seaborn-whitegrid", "ggplot")
        if style in available_styles
    ),
    "default",
)
plt.style.use(plot_style)
np.set_printoptions(precision=4, suppress=True)


## 1. Research contract

The fork is the start state. An intended EAST action expresses the corridor
policy; SOUTH expresses the shelter policy. NORTH, WEST, and exact ties stay
visible as `other` rather than being silently forced into either route.

The primary theoretical estimand is

$$
\Delta^*(p)=Q^*(s_0,\mathrm{EAST})-Q^*(s_0,\mathrm{SOUTH}),
$$

and its zero crossing $p^*$. The primary learned estimand is the fraction of
independent training seeds whose final greedy action selects EAST. Training
seeds—not episodes and not repeated held-out rollouts—are the independent
units.

Equal interaction budgets matter here. Shelter episodes are longer, so an
equal episode count would give shelter-taking methods more updates.

Every learner also starts with `Q(s,a)=8` for every state and action. This
route-neutral optimism is a declared coverage device, not a guess that one
route is better. Without it, an early random preference can make the long
alternative route exponentially hard to rediscover under epsilon-greedy
control. The exact oracles are unchanged; the initialization only affects
how efficiently the learners gather evidence about both routes.


In [ ]:
research_contract = pd.DataFrame(
    [
        {
            "claim": "In the declared high-reliability domain, the greedy optimum has one algorithm-independent EAST/SOUTH boundary.",
            "evidence": "exact start-action gap and numerical zero crossing",
        },
        {
            "claim": "Persistent exploration changes the continuation policy being valued.",
            "evidence": "greedy versus epsilon-soft exact boundary",
        },
        {
            "claim": "Q-learning and on-policy backups can approach different persistent-epsilon boundaries.",
            "evidence": "seed-level learned corridor probability",
        },
        {
            "claim": "Expected SARSA should mainly reduce target variance relative to SARSA.",
            "evidence": "boundary agreement plus a controlled exact backup-variance decomposition",
        },
    ]
)
display(research_contract)


## 2. Two kinds of noise, two exact objectives

Reliability $p<1$ belongs to the environment: an intended action can be
executed incorrectly even after learning ends. Epsilon belongs to the
behavior policy: with probability $\epsilon$ the agent deliberately samples
an action from the uniform distribution.

Q-learning bootstraps with $\max_a Q(s',a)$ and therefore targets greedy
control. SARSA samples the next exploratory action. Expected SARSA replaces
that sample by its conditional expectation. With persistent epsilon, the
exact on-policy control operator is

$$
V_\epsilon(s)=(1-\epsilon)\max_a Q(s,a)
  +\epsilon\frac{1}{|\mathcal A|}\sum_a Q(s,a).
$$

Turning exploration off after training and continuing to explore online are
therefore different deployment questions. We evaluate both without allowing
either evaluation clone to update.


In [ ]:
def corridor_environment(
    hazard_mode: str,
    reliability: float,
) -> RiskyCorridorEnv:
    return RiskyCorridorEnv(
        corridor_reliability=float(reliability),
        hazard_mode=hazard_mode,
        recoverable_hazard_penalty=RECOVERABLE_PENALTY,
        lethal_hazard_penalty=LETHAL_PENALTY,
    )

diagram_env = corridor_environment("lethal", 0.90)
assert diagram_env.fork_state == diagram_env.start_state
assert diagram_env.shelter_states.isdisjoint(diagram_env.hazard_positions)
assert all(edge in diagram_env.static_walls for edge in diagram_env.shelter_walls)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_maze(diagram_env, ax=axes[0], title="One topology, two consequence laws")
axes[0].plot(
    [0, 8], [1, 1], color="#D55E00", linewidth=3, alpha=0.72, label="exposed corridor"
)
axes[0].plot(
    [0, 0, 8, 8], [1, 3, 3, 1],
    color="#0072B2", linewidth=3, alpha=0.72, label="protected shelter",
)
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False)
plot_transition_noise(
    diagram_env,
    ax=axes[1],
    title="Same actuator reliability on both routes",
)
plt.tight_layout()
plt.show()


A recoverable encounter costs `-0.50`, a little more than eight ordinary
movement costs, and the episode continues. A lethal encounter costs `-8`
and terminates, forfeiting the chance to collect the `+8` goal. This is a
qualitative change in the process, not merely a cosmetic change of scale.


In [ ]:
def oracle_solution(hazard_mode: str, reliability: float, epsilon: float):
    env = corridor_environment(hazard_mode, reliability)
    model = env.exact_mdp()
    if epsilon == 0.0:
        solution = value_iteration(model, gamma=GAMMA)
        greedy_policy = solution.policy
    else:
        solution = epsilon_soft_value_iteration(
            model,
            epsilon=epsilon,
            gamma=GAMMA,
        )
        greedy_policy = solution.greedy_policy
    if not solution.converged:
        raise RuntimeError("Exact value iteration did not converge")
    return env, solution, greedy_policy

def exact_gap(hazard_mode: str, reliability: float, epsilon: float) -> float:
    env, solution, _ = oracle_solution(hazard_mode, reliability, epsilon)
    start = env.state_to_index[env.fork_state]
    return float(
        solution.q_values[start, int(env.corridor_action)]
        - solution.q_values[start, int(env.shelter_action)]
    )

def crossing(x: np.ndarray, y: np.ndarray) -> float:
    indices = np.flatnonzero((y[:-1] <= 0.0) & (y[1:] > 0.0))
    if not len(indices):
        return float("nan")
    index = int(indices[0])
    weight = -y[index] / (y[index + 1] - y[index])
    return float(x[index] + weight * (x[index + 1] - x[index]))

oracle_points = 41 if SMOKE else (301 if QUICK else 1_001)
oracle_grid = np.linspace(0.55, 1.0, oracle_points)
oracle_rows = []
for hazard_mode in ("recoverable", "lethal"):
    for epsilon in (0.0, 0.01, PERSISTENT_EPSILON):
        for reliability in oracle_grid:
            oracle_rows.append(
                {
                    "hazard_mode": hazard_mode,
                    "epsilon": epsilon,
                    "reliability": float(reliability),
                    "start_action_gap": exact_gap(
                        hazard_mode,
                        float(reliability),
                        epsilon,
                    ),
                }
            )
oracle_frame = pd.DataFrame(oracle_rows)
threshold_rows = []
for (hazard_mode, epsilon), sample in oracle_frame.groupby(
    ["hazard_mode", "epsilon"], sort=False
):
    threshold_rows.append(
        {
            "hazard_mode": hazard_mode,
            "epsilon": epsilon,
            "threshold": crossing(
                sample["reliability"].to_numpy(),
                sample["start_action_gap"].to_numpy(),
            ),
            "gap_at_perfect_control": float(sample.iloc[-1]["start_action_gap"]),
        }
    )
exact_thresholds = pd.DataFrame(threshold_rows)
display(exact_thresholds)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
epsilon_styles = {0.0: "-", 0.01: "--", PERSISTENT_EPSILON: ":"}
for ax, hazard_mode in zip(axes, ("recoverable", "lethal"), strict=True):
    sample = oracle_frame.loc[oracle_frame["hazard_mode"].eq(hazard_mode)]
    if hazard_mode == "lethal":
        sample = sample.loc[sample["reliability"].ge(0.965)]
    for epsilon, line in sample.groupby("epsilon", sort=True):
        label = "greedy oracle" if epsilon == 0 else f"epsilon-soft, ε={epsilon:g}"
        ax.plot(
            line["reliability"],
            line["start_action_gap"],
            linestyle=epsilon_styles[float(epsilon)],
            linewidth=2.2,
            label=label,
        )
    ax.axhline(0.0, color="black", linewidth=1)
    ax.set(
        xlabel="intended-action reliability p",
        ylabel="exact EAST - SOUTH action value",
        title=f"{hazard_mode.title()} hazards",
    )
    ax.legend(frameon=False)
    ax.grid(alpha=0.22)
plt.tight_layout()
plt.show()


## 3. Matched, interaction-budgeted learning experiment

Exact roots determined where the grids needed resolution; small pilot runs
then calibrated coverage, alpha, and the 100,000-interaction budget. The
resulting YAML files are the pre-specified full-run design—not a claim that
this pilot-informed study was preregistered. The recoverable grid resolves
both theoretical transitions. The lethal grid magnifies the narrow greedy
transition near perfect control. Every method receives the same interaction
budget, root-seed panel, alpha, gamma, and epsilon, with isolated random
streams.

The two held-out scenarios share environment seeds:

- `frozen_greedy`: exploration is disabled;
- `continuing_behavior`: the learned exploration schedule is sampled but no
  updates are performed.


In [ ]:
FULL_CONFIG_PATHS = {
    "recoverable": REPO_ROOT / "configs" / "shortcut_or_shelter_recoverable.yaml",
    "lethal": REPO_ROOT / "configs" / "shortcut_or_shelter_lethal.yaml",
}
SMOKE_GRIDS = {
    "recoverable": (0.79, 0.83),
    "lethal": (0.985, 1.0),
}
QUICK_GRIDS = {
    "recoverable": (0.70, 0.79, 0.805, 0.825, 0.84, 0.85, 0.86, 0.90, 1.0),
    "lethal": (0.970, 0.982, 0.987, 0.989, 0.991, 0.995, 1.0),
}

def configured_main_experiment(hazard_mode: str) -> ExperimentConfig:
    full = ExperimentConfig.from_yaml(FULL_CONFIG_PATHS[hazard_mode])
    if not (SMOKE or QUICK):
        return full
    grid = SMOKE_GRIDS[hazard_mode] if SMOKE else QUICK_GRIDS[hazard_mode]
    seeds = (0,) if SMOKE else tuple(range(4))
    interaction_steps = 80 if SMOKE else 2_500
    return replace(
        full,
        seeds=seeds,
        sweep={"environment.corridor_reliability": tuple(grid)},
        total_interaction_steps=interaction_steps,
        snapshot_interval=1_000_000,
        snapshot_step_interval=20 if SMOKE else 625,
        policy_evaluation=replace(
            full.policy_evaluation,
            interval_episodes=1_000_000,
            episodes_per_checkpoint=1 if SMOKE else 4,
            include_initial=False,
            include_final=True,
        ),
        execution=ExecutionSpec(parallel_workers=1),
        artifacts=replace(
            full.artifacts,
            output_dir=RESULTS_DIR,
            flush_rows=200 if SMOKE else 5_000,
        ),
    )

main_configs = {
    hazard_mode: configured_main_experiment(hazard_mode)
    for hazard_mode in ("recoverable", "lethal")
}
assert all(
    float(agent.parameters["initial_q"]) == INITIAL_Q
    and float(agent.parameters["epsilon"]) == PERSISTENT_EPSILON
    and float(agent.parameters["learning_rate"]) == 0.05
    for config in main_configs.values()
    for agent in config.agents
)
preflight = pd.DataFrame(
    [
        {"hazard_mode": hazard_mode, **estimate_run(config).as_dict()}
        for hazard_mode, config in main_configs.items()
    ]
)
display(preflight)


In [ ]:
def run_or_reopen(label: str, config: ExperimentConfig) -> ExperimentResult:
    existing = EXISTING_RUNS[label]
    if existing is not None:
        path = Path(existing)
        path = path if path.is_absolute() else REPO_ROOT / path
        store = RunStore.open(path)
        if store.manifest.experiment_name != config.name:
            raise ValueError(
                f"{label} run is {store.manifest.experiment_name!r}, expected {config.name!r}"
            )
        print(f"Reopened {label}: {store.run_directory}")
        return ExperimentResult(
            experiment_id=store.manifest.run_id,
            run_directory=store.run_directory,
            metadata={"reopened": True},
        )
    print(f"Starting {label}: {len(config.trials())} matched trials")
    result = Experiment(config).run(persist=True, progress=SHOW_PROGRESS)
    print(f"Completed {label}: {result.run_directory}")
    return result

main_results = {
    hazard_mode: run_or_reopen(hazard_mode, config)
    for hazard_mode, config in main_configs.items()
}


In [ ]:
def trial_design(config: ExperimentConfig) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "trial_id": trial.trial_id,
                "seed": trial.seed,
                "agent": trial.agent.name,
                "hazard_mode": trial.environment.parameters["hazard_mode"],
                "reliability": float(
                    trial.environment.parameters.get(
                        "corridor_reliability",
                        trial.environment.parameters.get("action_reliability", 1.0),
                    )
                ),
            }
            for trial in config.trials()
        ]
    )

def concatenate_tables(frames: list[pd.DataFrame]) -> pd.DataFrame:
    # Parquet schemas can contain diagnostics that are entirely missing in
    # one hazard condition. Drop only per-frame all-missing columns before
    # concatenation; this avoids dtype guessing warnings while preserving a
    # column wherever at least one condition actually observed it.
    return pd.concat(
        [frame.dropna(axis=1, how="all") for frame in frames],
        ignore_index=True,
        sort=False,
    )

main_design = concatenate_tables(
    [trial_design(config) for config in main_configs.values()],
)
main_training = concatenate_tables(
    [result.training_episodes for result in main_results.values()],
)
main_evaluations = concatenate_tables(
    [result.evaluations for result in main_results.values()],
)
assert set(main_design["agent"]) == set(METHOD_ORDER)
assert main_design["trial_id"].is_unique
assert set(main_evaluations["evaluation_policy_mode"]) == {"greedy", "behavior"}
paired_panels = main_evaluations.pivot_table(
    index=["trial_id", "evaluation_episode"],
    columns="evaluation_policy_mode",
    values="evaluation_seed",
    aggfunc="first",
).dropna()
assert (paired_panels["greedy"] == paired_panels["behavior"]).all()
final_steps = main_training.groupby("trial_id")["observed_step_count"].max()
expected_steps = {
    trial.trial_id: trial.total_interaction_steps
    for config in main_configs.values()
    for trial in config.trials()
}
assert all(int(final_steps[trial_id]) == budget for trial_id, budget in expected_steps.items())
print(
    "trials / training episodes / held-out episodes:",
    len(main_design), len(main_training), len(main_evaluations),
)


In [ ]:
def classify_action(q_row: np.ndarray, env: RiskyCorridorEnv) -> tuple[str, int]:
    maximizers = np.flatnonzero(np.isclose(q_row, np.max(q_row), rtol=1e-10, atol=1e-12))
    if len(maximizers) != 1:
        return "tie", -1
    action = int(maximizers[0])
    if action == int(env.corridor_action):
        return "corridor", action
    if action == int(env.shelter_action):
        return "shelter", action
    return "other", action

def final_policy_rows(
    result: ExperimentResult,
    config: ExperimentConfig,
) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    design = trial_design(config)
    final = (
        result.snapshots.query("episode >= 0")
        .sort_values(["trial_id", "global_step", "episode"])
        .groupby("trial_id", as_index=False)
        .tail(1)
        .merge(design, on=["trial_id", "seed", "agent"], validate="one_to_one")
    )
    rows = []
    tables: dict[str, np.ndarray] = {}
    for row in final.itertuples(index=False):
        table = result.q_snapshots(row.trial_id, keys=(row.snapshot_key,))[row.snapshot_key]
        tables[row.trial_id] = table
        env = corridor_environment(row.hazard_mode, row.reliability)
        start = env.state_to_index[env.fork_state]
        choice, action = classify_action(table[start], env)
        model = env.exact_mdp()
        learned_policy = np.argmax(table, axis=1)
        learned_values = exact_policy_evaluation(
            model,
            learned_policy,
            gamma=GAMMA,
            method="direct",
        ).values
        optimum = value_iteration(model, gamma=GAMMA)
        learned_gap = float(
            table[start, int(env.corridor_action)]
            - table[start, int(env.shelter_action)]
        )
        target_epsilon = 0.0 if row.agent == "q_learning" else PERSISTENT_EPSILON
        target_gap = exact_gap(row.hazard_mode, row.reliability, target_epsilon)
        rows.append(
            {
                "trial_id": row.trial_id,
                "seed": row.seed,
                "agent": row.agent,
                "hazard_mode": row.hazard_mode,
                "reliability": row.reliability,
                "global_step": row.global_step,
                "choice": choice,
                "greedy_action": action,
                "corridor_selected": float(choice == "corridor"),
                "start_action_gap": learned_gap,
                "target_oracle_gap": target_gap,
                "absolute_gap_error": abs(learned_gap - target_gap),
                "exact_deployment_regret": max(
                    0.0,
                    float(optimum.values[start] - learned_values[start]),
                ),
            }
        )
    return pd.DataFrame(rows), tables

final_frames = []
final_q_tables: dict[str, np.ndarray] = {}
for hazard_mode in ("recoverable", "lethal"):
    frame, tables = final_policy_rows(
        main_results[hazard_mode],
        main_configs[hazard_mode],
    )
    final_frames.append(frame)
    final_q_tables.update(tables)
final_choices = pd.concat(final_frames, ignore_index=True)
display(
    final_choices.groupby(["hazard_mode", "agent", "choice"], as_index=False)
    .size()
    .rename(columns={"size": "training_seeds"})
)
unresolved = final_choices["choice"].isin(["other", "tie"])
unresolved_rate = float(unresolved.mean())
gap_calibration = (
    final_choices.groupby(["hazard_mode", "agent"], as_index=False)
    .agg(
        mean_absolute_gap_error=("absolute_gap_error", "mean"),
        median_absolute_gap_error=("absolute_gap_error", "median"),
    )
)
display(gap_calibration)
if unresolved.any():
    message = (
        f"{unresolved.sum()} of {len(final_choices)} final policies "
        f"({unresolved_rate:.1%}) have a tied or off-route fork action."
    )
    if not (SMOKE or QUICK):
        raise RuntimeError(
            message + " The publication run is under-resolved; do not interpret its boundary."
        )
    print("UNDERTRAINING WARNING:", message, "QUICK/SMOKE output is mechanics-only.")

endpoint_rows = []
endpoint_expectations = (
    ("recoverable", "min", METHOD_ORDER, 0.0),
    ("recoverable", "max", METHOD_ORDER, 1.0),
    ("lethal", "min", METHOD_ORDER, 0.0),
    ("lethal", "max", ("q_learning",), 1.0),
    ("lethal", "max", ("sarsa", "expected_sarsa"), 0.0),
)
for hazard_mode, endpoint, agents, expected in endpoint_expectations:
    mode_rows = final_choices.loc[final_choices["hazard_mode"].eq(hazard_mode)]
    reliability = (
        float(mode_rows["reliability"].min())
        if endpoint == "min"
        else float(mode_rows["reliability"].max())
    )
    for agent in agents:
        observed = float(
            mode_rows.loc[
                mode_rows["agent"].eq(agent)
                & mode_rows["reliability"].eq(reliability),
                "corridor_selected",
            ].mean()
        )
        endpoint_rows.append(
            {
                "hazard_mode": hazard_mode,
                "reliability": reliability,
                "agent": agent,
                "expected_corridor_fraction": expected,
                "observed_corridor_fraction": observed,
                "passes": observed >= 0.8 if expected == 1.0 else observed <= 0.2,
            }
        )
endpoint_calibration = pd.DataFrame(endpoint_rows)
display(endpoint_calibration)
if RUN_PROFILE == "FULL" and not endpoint_calibration["passes"].all():
    raise RuntimeError(
        "The full run failed its pre-specified endpoint calibration; "
        "do not interpret the interior route boundary."
    )


In [ ]:
def binary_seed_summary(
    frame: pd.DataFrame,
    *,
    value: str,
    groups: tuple[str, ...],
) -> pd.DataFrame:
    z = 1.959963984540054
    rows = []
    for keys, sample in frame.groupby(list(groups), dropna=False, sort=True):
        key_tuple = keys if isinstance(keys, tuple) else (keys,)
        values = sample[value].to_numpy(dtype=float)
        if not np.isin(values, (0.0, 1.0)).all():
            raise ValueError(f"{value} must be binary for a Wilson interval")
        n = len(values)
        proportion = float(values.mean())
        denominator = 1.0 + z**2 / n
        center = (proportion + z**2 / (2.0 * n)) / denominator
        radius = (
            z
            * np.sqrt(proportion * (1.0 - proportion) / n + z**2 / (4.0 * n**2))
            / denominator
        )
        rows.append(
            {
                **dict(zip(groups, key_tuple, strict=True)),
                "mean": proportion,
                "ci_low": max(0.0, center - radius),
                "ci_high": min(1.0, center + radius),
                "n_seeds": int(sample["seed"].nunique()),
            }
        )
    return pd.DataFrame(rows)

def continuous_seed_summary(
    frame: pd.DataFrame,
    *,
    value: str,
    groups: tuple[str, ...],
    seed: int,
) -> pd.DataFrame:
    rows = []
    for keys, sample in frame.groupby(list(groups), dropna=False, sort=True):
        key_tuple = keys if isinstance(keys, tuple) else (keys,)
        values = sample[value].to_numpy(dtype=float)
        low, high = bootstrap_confidence_interval(
            values,
            n_resamples=N_RESAMPLES,
            seed=seed,
        )
        rows.append(
            {
                **dict(zip(groups, key_tuple, strict=True)),
                "mean": float(values.mean()),
                "ci_low": low,
                "ci_high": high,
                "n_seeds": int(sample["seed"].nunique()),
            }
        )
    return pd.DataFrame(rows)

corridor_summary = binary_seed_summary(
    final_choices,
    value="corridor_selected",
    groups=("hazard_mode", "agent", "reliability"),
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
fig.suptitle("Learned route selection" + EMPIRICAL_TITLE, fontweight="bold")
for ax, hazard_mode in zip(axes, ("recoverable", "lethal"), strict=True):
    sample = corridor_summary.loc[corridor_summary["hazard_mode"].eq(hazard_mode)]
    for agent in METHOD_ORDER:
        line = sample.loc[sample["agent"].eq(agent)].sort_values("reliability")
        ax.plot(
            line["reliability"], line["mean"], marker="o",
            color=METHOD_COLORS[agent], label=METHOD_LABELS[agent],
        )
        ax.fill_between(
            line["reliability"], line["ci_low"], line["ci_high"],
            color=METHOD_COLORS[agent], alpha=0.14,
        )
    oracle = exact_thresholds.loc[
        exact_thresholds["hazard_mode"].eq(hazard_mode)
        & exact_thresholds["epsilon"].eq(0.0),
        "threshold",
    ].iloc[0]
    ax.axvline(oracle, color="black", linestyle="--", label="greedy exact boundary")
    soft = exact_thresholds.loc[
        exact_thresholds["hazard_mode"].eq(hazard_mode)
        & exact_thresholds["epsilon"].eq(PERSISTENT_EPSILON),
        "threshold",
    ].iloc[0]
    if np.isfinite(soft):
        ax.axvline(soft, color="0.35", linestyle=":", label="epsilon-soft boundary")
    else:
        ax.text(
            0.02, 0.04, "epsilon-soft oracle never selects corridor",
            transform=ax.transAxes, fontsize=9,
        )
    ax.set(
        title=f"{hazard_mode.title()} hazards",
        xlabel="intended-action reliability p",
        ylabel="fraction of training seeds selecting corridor",
        ylim=(-0.04, 1.04),
    )
    ax.grid(alpha=0.22)
    ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()


The exact oracle changes action discretely. The empirical transition is
softened because independently trained finite-sample Q tables disagree near
a small action gap. A smooth-looking seed proportion is uncertainty across
learned policies, not a stochastic mixture chosen by one greedy policy.

Protocol-v2's generic `q_error` fields always use the greedy exact MDP. We do
not use those fields to score SARSA or Expected SARSA against an objective
they did not target. `target_oracle_gap` above is greedy for Q-learning and
epsilon-soft for the two on-policy methods; `exact_deployment_regret` asks a
separate question about the final greedy policy after exploration is turned
off.


In [ ]:
def checkpoint_policy_rows(
    result: ExperimentResult,
    config: ExperimentConfig,
    targets: tuple[float, ...] = (0.25, 0.50, 0.75, 1.00),
) -> pd.DataFrame:
    design = trial_design(config)
    snapshots = result.snapshots.query("episode >= 0").merge(
        design,
        on=["trial_id", "seed", "agent"],
        validate="many_to_one",
    )
    rows = []
    budget = int(config.total_interaction_steps or 0)
    if budget <= 0:
        raise ValueError("checkpoint boundary analysis requires an interaction budget")
    for trial_id, sample in snapshots.groupby("trial_id", sort=False):
        sample = sample.sort_values("global_step")
        selected = []
        for target in targets:
            eligible = sample.loc[sample["global_step"].le(target * budget)]
            selected.append(eligible.iloc[-1] if len(eligible) else sample.iloc[0])
        keys = tuple(dict.fromkeys(str(row["snapshot_key"]) for row in selected))
        tables = result.q_snapshots(trial_id, keys=keys)
        for target, row in zip(targets, selected, strict=True):
            env = corridor_environment(row["hazard_mode"], float(row["reliability"]))
            start = env.state_to_index[env.fork_state]
            table = tables[str(row["snapshot_key"])]
            choice, _ = classify_action(table[start], env)
            rows.append(
                {
                    "trial_id": trial_id,
                    "seed": int(row["seed"]),
                    "agent": row["agent"],
                    "hazard_mode": row["hazard_mode"],
                    "reliability": float(row["reliability"]),
                    "progress": target,
                    "target_global_step": int(target * budget),
                    "observed_global_step": int(row["global_step"]),
                    "step_lag": int(target * budget) - int(row["global_step"]),
                    "corridor_selected": float(choice == "corridor"),
                }
            )
    return pd.DataFrame(rows)

checkpoint_choices = pd.concat(
    [
        checkpoint_policy_rows(main_results[mode], main_configs[mode])
        for mode in ("recoverable", "lethal")
    ],
    ignore_index=True,
)
assert checkpoint_choices["step_lag"].eq(0).all(), (
    "Boundary checkpoints must be captured at exact interaction counts; "
    "check snapshot_step_interval."
)
stability_wide = checkpoint_choices.loc[
    checkpoint_choices["progress"].isin([0.75, 1.0])
].pivot(
    index=["trial_id", "seed", "agent", "hazard_mode", "reliability"],
    columns="progress",
    values="corridor_selected",
)
stability_wide["changed_last_quarter"] = stability_wide[0.75] != stability_wide[1.0]
stability_summary = (
    stability_wide.reset_index()
    .groupby(["hazard_mode", "agent", "reliability"], as_index=False)
    .agg(
        changed_fraction=("changed_last_quarter", "mean"),
        n_seeds=("seed", "nunique"),
    )
)
display(stability_summary)

endpoint_stability = []
for hazard_mode in ("recoverable", "lethal"):
    sample = stability_summary.loc[stability_summary["hazard_mode"].eq(hazard_mode)]
    for reliability in (float(sample["reliability"].min()), float(sample["reliability"].max())):
        endpoint_stability.append(sample.loc[sample["reliability"].eq(reliability)])
endpoint_stability = pd.concat(endpoint_stability, ignore_index=True)
if RUN_PROFILE == "FULL" and endpoint_stability["changed_fraction"].gt(0.2).any():
    raise RuntimeError(
        "More than 20% of seeds changed route at a calibration endpoint in "
        "the final quarter; the finite-budget boundary is not stable enough to report."
    )
checkpoint_summary = binary_seed_summary(
    checkpoint_choices,
    value="corridor_selected",
    groups=("hazard_mode", "agent", "progress", "reliability"),
)

def empirical_half_boundary(sample: pd.DataFrame) -> tuple[float, int]:
    ordered = sample.sort_values("reliability")
    x = ordered["reliability"].to_numpy(dtype=float)
    y = ordered["mean"].to_numpy(dtype=float)
    violations = int(np.count_nonzero(np.diff(y) < -0.05))
    indices = np.flatnonzero(y >= 0.5)
    if not len(indices):
        return float("nan"), violations
    index = int(indices[0])
    if index == 0:
        return float(x[0]), violations
    if y[index] == y[index - 1]:
        return float(x[index]), violations
    weight = (0.5 - y[index - 1]) / (y[index] - y[index - 1])
    return float(x[index - 1] + weight * (x[index] - x[index - 1])), violations

def seed_block_boundary_bootstrap(
    frame: pd.DataFrame,
    *,
    n_resamples: int,
    seed: int,
) -> pd.DataFrame:
    seeds = np.sort(frame["seed"].unique())
    rng = np.random.default_rng(seed)
    rows = []
    for bootstrap_id in range(n_resamples):
        sampled_seeds = rng.choice(seeds, size=len(seeds), replace=True)
        blocks = []
        for block_id, sampled_seed in enumerate(sampled_seeds):
            block = frame.loc[frame["seed"].eq(sampled_seed)].copy()
            block["bootstrap_block"] = block_id
            blocks.append(block)
        sampled = pd.concat(blocks, ignore_index=True)
        curve = (
            sampled.groupby(["hazard_mode", "agent", "reliability"], as_index=False)[
                "corridor_selected"
            ]
            .mean()
            .rename(columns={"corridor_selected": "mean"})
        )
        for (hazard_mode, agent), panel in curve.groupby(
            ["hazard_mode", "agent"], sort=False
        ):
            boundary, violations = empirical_half_boundary(panel)
            rows.append(
                {
                    "bootstrap_id": bootstrap_id,
                    "hazard_mode": hazard_mode,
                    "agent": agent,
                    "half_boundary": boundary,
                    "right_censored": not np.isfinite(boundary),
                    "monotonicity_violations": violations,
                }
            )
    return pd.DataFrame(rows)

learned_boundary_rows = []
for keys, sample in checkpoint_summary.groupby(
    ["hazard_mode", "agent", "progress"], sort=False
):
    boundary, violations = empirical_half_boundary(sample)
    learned_boundary_rows.append(
        {
            "hazard_mode": keys[0],
            "agent": keys[1],
            "progress": keys[2],
            "half_boundary": boundary,
            "monotonicity_violations": violations,
        }
    )
learned_boundaries = pd.DataFrame(learned_boundary_rows)

boundary_bootstrap = seed_block_boundary_bootstrap(
    final_choices,
    n_resamples=N_RESAMPLES,
    seed=41,
)
final_boundary_rows = []
for (hazard_mode, agent), sample in boundary_bootstrap.groupby(
    ["hazard_mode", "agent"], sort=False
):
    finite = sample.loc[np.isfinite(sample["half_boundary"]), "half_boundary"]
    final_boundary_rows.append(
        {
            "hazard_mode": hazard_mode,
            "agent": agent,
            "median_half_boundary": float(finite.median()) if len(finite) else np.nan,
            "ci_low": float(finite.quantile(0.025)) if len(finite) else np.nan,
            "ci_high": float(finite.quantile(0.975)) if len(finite) else np.nan,
            "right_censored_fraction": float(sample["right_censored"].mean()),
        }
    )
final_boundary_summary = pd.DataFrame(final_boundary_rows)
observed_final_boundaries = learned_boundaries.loc[
    learned_boundaries["progress"].eq(1.0),
    ["hazard_mode", "agent", "half_boundary", "monotonicity_violations"],
].rename(columns={"half_boundary": "observed_half_boundary"})
final_boundary_summary = observed_final_boundaries.merge(
    final_boundary_summary,
    on=["hazard_mode", "agent"],
    validate="one_to_one",
)

boundary_wide = boundary_bootstrap.pivot(
    index=["bootstrap_id", "hazard_mode"],
    columns="agent",
    values="half_boundary",
)
contrast_rows = []
for hazard_mode, sample in boundary_wide.groupby(level="hazard_mode"):
    for comparison in ("sarsa", "expected_sarsa"):
        differences = (sample[comparison] - sample["q_learning"]).dropna()
        contrast_rows.append(
            {
                "hazard_mode": hazard_mode,
                "comparison": comparison,
                "median_boundary_shift_vs_q": float(differences.median())
                if len(differences)
                else np.nan,
                "ci_low": float(differences.quantile(0.025))
                if len(differences)
                else np.nan,
                "ci_high": float(differences.quantile(0.975))
                if len(differences)
                else np.nan,
                "complete_bootstrap_pairs": len(differences),
            }
        )
boundary_contrasts = pd.DataFrame(contrast_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle("Boundary over training" + EMPIRICAL_TITLE, fontweight="bold")
for ax, hazard_mode in zip(axes, ("recoverable", "lethal"), strict=True):
    sample = learned_boundaries.loc[learned_boundaries["hazard_mode"].eq(hazard_mode)]
    for agent in METHOD_ORDER:
        line = sample.loc[sample["agent"].eq(agent)].sort_values("progress")
        ax.plot(
            100 * line["progress"], line["half_boundary"], marker="o",
            color=METHOD_COLORS[agent], label=METHOD_LABELS[agent],
        )
    greedy_boundary = exact_thresholds.loc[
        exact_thresholds["hazard_mode"].eq(hazard_mode)
        & exact_thresholds["epsilon"].eq(0.0),
        "threshold",
    ].iloc[0]
    ax.axhline(greedy_boundary, color="black", linestyle="--", label="greedy oracle")
    soft_boundary = exact_thresholds.loc[
        exact_thresholds["hazard_mode"].eq(hazard_mode)
        & exact_thresholds["epsilon"].eq(PERSISTENT_EPSILON),
        "threshold",
    ].iloc[0]
    if np.isfinite(soft_boundary):
        ax.axhline(soft_boundary, color="0.35", linestyle=":", label="epsilon-soft oracle")
    ax.set(
        title=f"{hazard_mode.title()} learned boundary",
        xlabel="interaction budget completed (%)",
        ylabel="reliability at 50% corridor selection",
    )
    ax.grid(alpha=0.22)
    ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()
display(learned_boundaries)
display(final_boundary_summary)
display(boundary_contrasts)


We do not force these finite-seed curves to be monotone. A reported
`monotonicity_violations` count warns when a single crossover would be a poor
summary. Missing boundaries are right-censored: fewer than half of the seed
panel selected the corridor anywhere on the pre-specified full-run grid.
Pointwise binary ribbons use Wilson score intervals. Boundary intervals and
method contrasts resample each root seed as one matched block across every
reliability and algorithm; episodes never become fake replicates.


In [ ]:
def evaluation_trial_summary(frame: pd.DataFrame) -> pd.DataFrame:
    grouped = [
        "trial_id", "agent", "seed", "env_hazard_mode", "env_action_reliability",
        "evaluation_policy_mode",
    ]
    result = (
        frame.groupby(grouped, as_index=False)
        .agg(
            episode_return=("episode_return", "mean"),
            success=("success", "mean"),
            failure=("failure", "mean"),
            total_episode_steps=("episode_length", "sum"),
            hazard_penalty_steps=("env_hazard_penalty_steps", "sum"),
            realized_corridor=(
                "env_realized_route",
                lambda values: float(np.mean(np.asarray(values) == "corridor")),
            ),
        )
        .rename(
            columns={
                "env_hazard_mode": "hazard_mode",
                "env_action_reliability": "reliability",
            }
        )
    )
    result["penalty_steps_per_1000"] = (
        1_000.0 * result["hazard_penalty_steps"] / result["total_episode_steps"]
    )
    return result

evaluation_trials = evaluation_trial_summary(main_evaluations)
regret_summary = continuous_seed_summary(
    final_choices,
    value="exact_deployment_regret",
    groups=("hazard_mode", "agent", "reliability"),
    seed=53,
)
return_summary = continuous_seed_summary(
    evaluation_trials,
    value="episode_return",
    groups=("hazard_mode", "agent", "reliability", "evaluation_policy_mode"),
    seed=59,
)
fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), constrained_layout=True)
fig.suptitle("Consequences" + EMPIRICAL_TITLE, fontweight="bold")
for row_index, hazard_mode in enumerate(("recoverable", "lethal")):
    for agent in METHOD_ORDER:
        line = regret_summary.loc[
            regret_summary["hazard_mode"].eq(hazard_mode)
            & regret_summary["agent"].eq(agent)
        ]
        axes[row_index, 0].plot(
            line["reliability"], line["mean"], marker="o",
            color=METHOD_COLORS[agent], label=METHOD_LABELS[agent],
        )
        axes[row_index, 0].fill_between(
            line["reliability"], line["ci_low"], line["ci_high"],
            color=METHOD_COLORS[agent], alpha=0.12,
        )

    for agent in METHOD_ORDER:
        for policy_mode, linestyle in (("greedy", "-"), ("behavior", "--")):
            line = return_summary.loc[
                return_summary["hazard_mode"].eq(hazard_mode)
                & return_summary["agent"].eq(agent)
                & return_summary["evaluation_policy_mode"].eq(policy_mode)
            ]
            axes[row_index, 1].plot(
                line["reliability"], line["mean"],
                color=METHOD_COLORS[agent], linestyle=linestyle,
                marker="o" if policy_mode == "greedy" else None,
                label=f"{METHOD_LABELS[agent]} — {policy_mode}",
            )
            axes[row_index, 1].fill_between(
                line["reliability"], line["ci_low"], line["ci_high"],
                color=METHOD_COLORS[agent], alpha=0.08,
            )
    axes[row_index, 0].set(
        title=f"{hazard_mode.title()}: exact regret of deployed greedy policy",
        xlabel="reliability p", ylabel="start-state regret",
    )
    axes[row_index, 1].set(
        title=f"{hazard_mode.title()}: held-out return",
        xlabel="reliability p", ylabel="episode return",
    )
    for ax in axes[row_index]:
        ax.grid(alpha=0.22)
        ax.legend(frameon=False, fontsize=8)
plt.show()

exposure_summary = continuous_seed_summary(
    evaluation_trials,
    value="penalty_steps_per_1000",
    groups=("hazard_mode", "agent", "reliability", "evaluation_policy_mode"),
    seed=61,
)
failure_summary = continuous_seed_summary(
    evaluation_trials,
    value="failure",
    groups=("hazard_mode", "agent", "reliability", "evaluation_policy_mode"),
    seed=67,
).rename(
    columns={
        "mean": "failure_mean",
        "ci_low": "failure_ci_low",
        "ci_high": "failure_ci_high",
    }
)
exposure_table = exposure_summary.merge(
    failure_summary[
        [
            "hazard_mode", "agent", "reliability", "evaluation_policy_mode",
            "failure_mean", "failure_ci_low", "failure_ci_high",
        ]
    ],
    on=["hazard_mode", "agent", "reliability", "evaluation_policy_mode"],
    validate="one_to_one",
)
exposure_table["method"] = exposure_table["agent"].map(METHOD_LABELS)
key_exposure = pd.concat(
    [
        exposure_table.loc[
            exposure_table["hazard_mode"].eq(hazard_mode)
            & exposure_table["reliability"].eq(reliability)
        ]
        for hazard_mode, reliability in DISAGREEMENT_POINTS.items()
    ],
    ignore_index=True,
)
display(
    key_exposure[
        [
            "hazard_mode", "reliability", "method", "evaluation_policy_mode",
            "mean", "ci_low", "ci_high",
            "failure_mean", "failure_ci_low", "failure_ci_high",
        ]
    ].rename(columns={"mean": "penalty_steps_per_1000"})
)


### The variance claim gets a controlled probe

Pooling empirical TD errors across a learned trajectory would confound the
backup rule with different state visits, actions, and policies. Instead we
hold one state-action pair, one exact epsilon-soft Q table, and one transition
kernel fixed. We then enumerate the one-step target distribution exactly.
SARSA samples the next action; Expected SARSA integrates it out. Their target
means must agree, and the variance difference is precisely the removable
next-action sampling component.


In [ ]:
def exact_backup_moments(
    model,
    q_values: np.ndarray,
    policy: np.ndarray,
    *,
    state: int,
    action: int,
    integrate_next_action: bool,
) -> tuple[float, float]:
    probabilities = []
    targets = []
    for next_state, transition_probability in enumerate(model.P[state, action]):
        if transition_probability == 0.0:
            continue
        reward = float(model.R[state, action, next_state])
        if model.terminal[next_state]:
            probabilities.append(float(transition_probability))
            targets.append(reward)
        elif integrate_next_action:
            probabilities.append(float(transition_probability))
            targets.append(
                reward
                + GAMMA * float(np.dot(policy[next_state], q_values[next_state]))
            )
        else:
            for next_action, action_probability in enumerate(policy[next_state]):
                if action_probability == 0.0:
                    continue
                probabilities.append(float(transition_probability * action_probability))
                targets.append(reward + GAMMA * float(q_values[next_state, next_action]))
    weights = np.asarray(probabilities, dtype=float)
    values = np.asarray(targets, dtype=float)
    weights /= weights.sum()
    mean = float(np.dot(weights, values))
    variance = float(np.dot(weights, np.square(values - mean)))
    return mean, variance

backup_variance_rows = []
for hazard_mode, reliability in DISAGREEMENT_POINTS.items():
    env, solution, _ = oracle_solution(
        hazard_mode,
        reliability,
        PERSISTENT_EPSILON,
    )
    model = env.exact_mdp()
    state = env.state_to_index[env.fork_state]
    action = int(env.corridor_action)
    expected_mean, expected_variance = exact_backup_moments(
        model,
        solution.q_values,
        solution.policy,
        state=state,
        action=action,
        integrate_next_action=True,
    )
    sampled_mean, sampled_variance = exact_backup_moments(
        model,
        solution.q_values,
        solution.policy,
        state=state,
        action=action,
        integrate_next_action=False,
    )
    np.testing.assert_allclose(sampled_mean, expected_mean, atol=1e-12)
    backup_variance_rows.append(
        {
            "hazard_mode": hazard_mode,
            "reliability": reliability,
            "target_mean": expected_mean,
            "expected_sarsa_variance": expected_variance,
            "sarsa_variance": sampled_variance,
            "next_action_sampling_component": sampled_variance - expected_variance,
        }
    )
backup_variance = pd.DataFrame(backup_variance_rows)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.1), constrained_layout=True)
for ax, row in zip(axes, backup_variance.itertuples(index=False), strict=True):
    ax.bar(
        ["Expected SARSA", "SARSA"],
        [row.expected_sarsa_variance, row.sarsa_variance],
        color=[METHOD_COLORS["expected_sarsa"], METHOD_COLORS["sarsa"]],
    )
    ax.set(
        title=f"{row.hazard_mode.title()} | p={row.reliability:g}",
        ylabel="exact one-step target variance",
    )
    ax.grid(axis="y", alpha=0.22)
plt.show()
display(backup_variance)


In [ ]:
def modal_policy(tables: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    policies = np.stack([np.argmax(table, axis=1) for table in tables])
    modes = np.array(
        [
            np.bincount(policies[:, state], minlength=tables[0].shape[1]).argmax()
            for state in range(policies.shape[1])
        ],
        dtype=int,
    )
    return modes, np.mean(policies == modes[None, :], axis=0)

fig, axes = plt.subplots(2, 5, figsize=(20, 7.5), constrained_layout=True)
fig.suptitle("Exact and learned policies" + EMPIRICAL_TITLE, fontweight="bold")
for row_index, hazard_mode in enumerate(("recoverable", "lethal")):
    available = np.sort(final_choices.loc[
        final_choices["hazard_mode"].eq(hazard_mode), "reliability"
    ].unique())
    reliability = float(available[np.argmin(np.abs(available - DISAGREEMENT_POINTS[hazard_mode]))])
    env, greedy, greedy_policy = oracle_solution(hazard_mode, reliability, 0.0)
    _, soft, soft_policy = oracle_solution(
        hazard_mode,
        reliability,
        PERSISTENT_EPSILON,
    )
    plot_policy(
        greedy_policy, env, ax=axes[row_index, 0],
        title=f"Greedy oracle\np={reliability:g}",
    )
    plot_policy(
        soft_policy, env, ax=axes[row_index, 1],
        title=f"ε-soft oracle\nε={PERSISTENT_EPSILON:g}",
    )
    for column, agent in enumerate(METHOD_ORDER, start=2):
        trial_ids = final_choices.loc[
            final_choices["hazard_mode"].eq(hazard_mode)
            & final_choices["reliability"].eq(reliability)
            & final_choices["agent"].eq(agent),
            "trial_id",
        ]
        policy, consensus = modal_policy([final_q_tables[trial_id] for trial_id in trial_ids])
        plot_policy(
            policy,
            env,
            values=consensus,
            vmin=0.0,
            vmax=1.0,
            cmap="Blues",
            colorbar_label="fraction choosing modal action",
            ax=axes[row_index, column],
            title=METHOD_LABELS[agent],
        )
    axes[row_index, 0].set_ylabel(hazard_mode.title())
plt.show()


In [ ]:
def greedy_rollout(
    table: np.ndarray,
    *,
    hazard_mode: str,
    reliability: float,
    seed: int,
) -> dict[str, object]:
    env = corridor_environment(hazard_mode, reliability)
    observation, _ = env.reset(seed=seed)
    trajectory = [env.agent_position]
    total_return = 0.0
    terminated = truncated = False
    while not (terminated or truncated):
        action = int(np.argmax(table[int(observation)]))
        observation, reward, terminated, truncated, _ = env.step(action)
        total_return += reward
        trajectory.append(env.agent_position)
    summary = env.episode_summary()
    env.close()
    return {
        "trajectory": trajectory,
        "episode_return": total_return,
        **summary,
    }

rollout_seed = 20_260_818
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
fig.suptitle("Fixed-seed rollouts" + EMPIRICAL_TITLE, fontweight="bold")
representative_rollouts = []
for row_index, hazard_mode in enumerate(("recoverable", "lethal")):
    available = np.sort(final_choices.loc[
        final_choices["hazard_mode"].eq(hazard_mode), "reliability"
    ].unique())
    reliability = float(available[np.argmin(np.abs(available - DISAGREEMENT_POINTS[hazard_mode]))])
    for column, agent in enumerate(METHOD_ORDER):
        candidates = final_choices.loc[
            final_choices["hazard_mode"].eq(hazard_mode)
            & final_choices["reliability"].eq(reliability)
            & final_choices["agent"].eq(agent)
            & final_choices["seed"].eq(REPRESENTATIVE_SEED)
        ]
        selected = candidates.iloc[0]
        rollout = greedy_rollout(
            final_q_tables[selected["trial_id"]],
            hazard_mode=hazard_mode,
            reliability=reliability,
            seed=rollout_seed,
        )
        representative_rollouts.append(
            {
                "hazard_mode": hazard_mode,
                "reliability": reliability,
                "agent": agent,
                **{key: value for key, value in rollout.items() if key != "trajectory"},
            }
        )
        env = corridor_environment(hazard_mode, reliability)
        plot_maze(
            env,
            trajectory=rollout["trajectory"],
            ax=axes[row_index, column],
            title=(
                f"{METHOD_LABELS[agent]} | R={rollout['episode_return']:.2f} | "
                f"{rollout['realized_route']}"
            ),
        )
        env.close()
    axes[row_index, 0].set_ylabel(f"{hazard_mode.title()} | p={reliability:g}")
plt.show()
display(pd.DataFrame(representative_rollouts))


These are not best-looking trajectories. They use training seed zero and one
fixed, declared rollout seed. The same environment seed does not force equal
transitions after policies choose different actions, but it prevents visual
examples from being selected post hoc by outcome.


## 4. Annealing exploration is a falsification check

Persistent exploration intentionally makes the on-policy target different
from greedy control. We therefore run a smaller sensitivity experiment at
disagreement points while epsilon decays toward zero. If the story above is
correct, SARSA, Expected SARSA, and Q-learning should move toward the same
greedy route given enough interactions. A failure to do so is evidence about
optimization, coverage, or budget—not evidence that the environment has
three different optimal policies.


In [ ]:
full_annealed_config = ExperimentConfig.from_yaml(
    REPO_ROOT / "configs" / "shortcut_or_shelter_annealed.yaml"
)
annealed_config = full_annealed_config
if SMOKE or QUICK:
    annealed_config = replace(
        full_annealed_config,
        seeds=(0,) if SMOKE else tuple(range(4)),
        total_interaction_steps=100 if SMOKE else 4_000,
        snapshot_interval=1_000_000,
        snapshot_step_interval=20 if SMOKE else 250,
        policy_evaluation=replace(
            full_annealed_config.policy_evaluation,
            interval_episodes=1_000_000,
            episodes_per_checkpoint=1 if SMOKE else 4,
            include_initial=False,
            include_final=True,
        ),
        execution=ExecutionSpec(parallel_workers=1),
        artifacts=replace(
            full_annealed_config.artifacts,
            output_dir=RESULTS_DIR,
            flush_rows=200 if SMOKE else 5_000,
        ),
    )
display(pd.Series(estimate_run(annealed_config).as_dict(), name="annealed preflight"))
annealed_result = run_or_reopen("annealed", annealed_config)
annealed_choices, annealed_q_tables = final_policy_rows(
    annealed_result,
    annealed_config,
)
annealed_summary = binary_seed_summary(
    annealed_choices,
    value="corridor_selected",
    groups=("hazard_mode", "agent", "reliability"),
)
annealed_summary["method"] = annealed_summary["agent"].map(METHOD_LABELS)
display(
    annealed_summary[
        ["hazard_mode", "reliability", "method", "mean", "ci_low", "ci_high", "n_seeds"]
    ]
)


## 5. What the evidence can and cannot say

**Supported interpretations**

- The route boundary belongs to a declared environment, reward law,
  discount, and policy class—not to an algorithm's personality.
- Lethal risk compounds across repeated exposed decisions, compressing the
  greedy transition near perfect execution.
- Persistent exploration can rationally move the epsilon-soft boundary or
  remove the corridor region entirely.
- SARSA and Expected SARSA should share the same expected persistent-epsilon
  target; their finite-sample variability need not match.

**Not supported**

- “SARSA is universally safer” or “Q-learning is reckless.”
- Treating held-out episodes as independent replicates.
- Calling a smoothed crossover ground truth when the exact action gap is
  available.
- Generalizing beyond this reward scale, topology, horizon, and exploration
  protocol without a sensitivity experiment.
- Extending the one EAST/SOUTH boundary below the plotted reliability range;
  at sufficiently poor control, NORTH or WEST can become exact-optimal.

The next notebook will add a visible CLEAR/STORM regime. Once regime is part
of state, stable switching is an ordinary augmented MDP with a policy
conditional on both position and weather. Hiding a persistent regime creates
a POMDP; a position-only Q table is then an information-limited baseline, not
the “real optimum.”


In [ ]:
provenance = pd.DataFrame(
    [
        {
            "study": label,
            "experiment": result.metadata.get("experiment_name", config.name),
            "run_directory": str(result.run_directory),
            "training_budget": config.total_interaction_steps,
            "training_seeds": len(config.seeds),
            "trials": len(config.trials()),
        }
        for label, result, config in (
            ("recoverable", main_results["recoverable"], main_configs["recoverable"]),
            ("lethal", main_results["lethal"], main_configs["lethal"]),
            ("annealed", annealed_result, annealed_config),
        )
    ]
)
display(provenance)
print("Canonical configs:")
for path in (*FULL_CONFIG_PATHS.values(), REPO_ROOT / "configs" / "shortcut_or_shelter_annealed.yaml"):
    print(" -", path.relative_to(REPO_ROOT))
